# 007: Attention Mechanism — Scaled dot-product attention derivation and implementation

## SCALED DOT-PRODUCT ATTENTION
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$
- $Q$ (query), $K$ (key), $V$ (value) are linear projections of the input.
- **Why scale by $\sqrt{d_k}$?** For large $d_k$, the dot products grow large, pushing softmax into regions with tiny gradients. Scaling stabilizes training.
---


In [ ]:
import numpy as np

def softmax(x, axis=-1):
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / np.sum(e_x, axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V, mask=None):
    """Computes scaled dot-product attention from scratch."""
    d_k = K.shape[-1]
    # Step 1: Compute attention scores
    scores = Q @ K.T / np.sqrt(d_k)  # (seq_q, seq_k)
    # Step 2: Apply mask (for causal / padding)
    if mask is not None:
        scores = scores + mask * (-1e9)
    # Step 3: Softmax to get attention weights
    weights = softmax(scores, axis=-1)
    # Step 4: Weighted sum of values
    output = weights @ V  # (seq_q, d_v)
    return output, weights



In [ ]:
print("--- Scaled Dot-Product Attention ---")
np.random.seed(42)
seq_len, d_model = 4, 8
X = np.random.randn(seq_len, d_model)

# Linear projections (simplified: use random weight matrices)
W_q = np.random.randn(d_model, d_model) * 0.1
W_k = np.random.randn(d_model, d_model) * 0.1
W_v = np.random.randn(d_model, d_model) * 0.1

Q = X @ W_q
K = X @ W_k
V = X @ W_v

output, weights = scaled_dot_product_attention(Q, K, V)

print("Attention Weights (each row sums to 1):")
print(weights.round(3))
print(f"\nOutput shape: {output.shape}")
print(f"Row sum check: {weights.sum(axis=-1).round(4)}")



In [ ]:
# ── Causal (autoregressive) masking ──
print("\n--- Causal Masked Attention ---")
causal_mask = np.triu(np.ones((seq_len, seq_len)), k=1)  # Upper triangular = 1
output_causal, weights_causal = scaled_dot_product_attention(Q, K, V, mask=causal_mask)
print("Causal Attention Weights:")
print(weights_causal.round(3))
print("[Insight] Each position can only attend to itself and previous positions.")
